In [223]:
import pandas as pd
import re
import nltk
import numpy as np
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from nltk.stem import PorterStemmer


from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


### Get all unique values for spell.csv the "type coulmn"

In [224]:
import pandas as pd
import re
from nltk.tokenize import word_tokenize



df = pd.read_csv("Spells.csv")
print(df.head())
print(df.columns.tolist())

   Spell ID       Incantation                        Spell Name  \
0         1             Accio                   Summoning Charm   
1         2         Aguamenti                Water-Making Spell   
2         3  Alarte Ascendare  Launch an object up into the air   
3         4         Alohomora                   Unlocking Charm   
4         5     Arania Exumai            Spider repelling spell   

                  Effect     Light  
0      Summons an object       NaN  
1         Conjures water  Icy blue  
2  Rockets target upward       Red  
3         Unlocks target      Blue  
4         Repels spiders      Blue  
['Spell ID', 'Incantation', 'Spell Name', 'Effect', 'Light']


# Show all unique spell names

In [225]:
# Show all unique spell names
unique_spell_names = df["Spell Name"].dropna().unique()

print(unique_spell_names)
print(f"\nTotal unique spell names: {len(unique_spell_names)}")

['Summoning Charm' 'Water-Making Spell' 'Launch an object up into the air'
 'Unlocking Charm' 'Spider repelling spell' 'Slowing Charm'
 'Killing Curse' 'Exploding Charm' 'Brackium Emendo' 'Cistem Aperio'
 'Locking Spell' 'Blasting Curse' 'Cruciatus Curse' 'Severing Charm'
 'Dissendium' 'Engorgement Charm' 'Episkey' 'Patronus Charm'
 'Disarming Charm' 'Expulso Curse' 'General Counter-Spell'
 'Human Presence Revealing Spell' 'Freezing Charm' 'Impediment Jinx'
 'Imperius Curse' 'Incarcerous Spell' 'Fire-Making Spell' 'Levicorpus'
 'Locomotion Charm' 'Leg-Locker Curse' 'Wand-Lighting Charm'
 'Lumos Maxima' 'Lumos Solem Spell' 'Muffliato Charm'
 'Wand-Extinguishing Charm' 'Memory Charm' 'Oculus Reparo' 'Oppugno Jinx'
 'Peskipiksi Pesternomi' 'Full Body-Bind Curse' 'Piertotum Locomotor'
 'Portus' 'Reverse Spell' 'Shield Charm' 'Protego Maxima'
 'Protego totalum' 'Shrinking Charm' 'Revulsion Jinx' 'Mending Charm'
 'Repello Inimicum' 'Muggle-Repelling Charm' 'Revelio Charm'
 'Tickling Charm' '

In [226]:
print(df.isnull().sum())

Spell ID        0
Incantation     0
Spell Name      0
Effect          0
Light          21
dtype: int64


In [227]:
# Combine text fields
# Include spell type if present so LDA can use explicit class signals where available.
type_col_candidates = [column for column in df.columns if str(column).strip().lower() == "type"]
type_series = df[type_col_candidates[0]].fillna("") if type_col_candidates else ""

df["text"] = (
    df["Spell Name"].fillna("") + " "
    + df["Effect"].fillna("") + " "
    + type_series
)

# Lowercase
df["text"] = df["text"].str.lower()

# Remove punctuation
df["text"] = df["text"].apply(
    lambda x: re.sub(r"[^a-zA-Z\s]", "", x)
)
import nltk

# Tokenization
df["tokens"] = df["text"].apply(word_tokenize)

# Standard English stopwords
stop_words = set(stopwords.words("english"))

# Harry Potter / dataset-specific stopwords
# Keep generic class words out so topics depend on distinctive terms.
custom_stopwords = {
    "spell",
    "charm",
    "curse",
    "jinx",
    "hex",
    "target",
    "object",
    "force",
    "forc",
    "maximum",
    "maxima",
    "wand",
    "turn",
    "injury",
    "injuries"
}

# Combine both sets
all_stopwords = stop_words.union(custom_stopwords)

# Remove stopwords
df["tokens"] = df["tokens"].apply(
    lambda words: [w for w in words if w not in all_stopwords]
)

stemmer = PorterStemmer()
lemmer = WordNetLemmatizer()

df["tokens"] = df["tokens"].apply(
    #lambda words: [stemmer.stem(w) for w in words]
    lambda words: [lemmer.lemmatize(w) for w in words]
)

# View results
print(df[["Spell Name", "tokens"]].head(20))

                          Spell Name                                    tokens
0                    Summoning Charm                      [summoning, summons]
1                 Water-Making Spell            [watermaking, conjures, water]
2   Launch an object up into the air             [launch, air, rocket, upward]
3                    Unlocking Charm                      [unlocking, unlocks]
4             Spider repelling spell       [spider, repelling, repels, spider]
5                      Slowing Charm  [slowing, slows, stop, target, velocity]
6                      Killing Curse           [killing, instantaneous, death]
7                    Exploding Charm             [exploding, small, explosion]
8                    Brackium Emendo            [brackium, emendo, mend, bone]
9                      Cistem Aperio             [cistem, aperio, open, chest]
10                     Locking Spell                     [locking, lock, door]
11                    Blasting Curse                

### Making a dictionary

In [228]:
dictionary = Dictionary(df["tokens"])
dictionary.filter_extremes(
    no_below=1,
    no_above=0.7
)
corpus = [dictionary.doc2bow(text) for text in df["tokens"]]

### Small model

In [229]:
# Guided LDA priors (still LDA): stronger bias for offensive/dark vs defensive/healing terms.
defensive_seed_words = ["heal", "heals", "healing", "episkey", "patronus", "shield", "protect", "protego", "counter", "mend", "defensive"]
offensive_seed_words = ["kill", "killing", "death", "cruciatus", "imperius", "imperio", "avada", "dark", "torture", "unforgivable", "pain", "victim"]

eta = np.full((4, len(dictionary)), 0.001, dtype=float)
for word in offensive_seed_words:
    word_id = dictionary.token2id.get(word)
    if word_id is not None:
        eta[0, word_id] = 2.0
for word in defensive_seed_words:
    word_id = dictionary.token2id.get(word)
    if word_id is not None:
        eta[1, word_id] = 2.0

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=4,
    random_state=42,
    passes=40,
    alpha="auto",
    eta=eta
)

In [230]:
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}: {topic}")

Topic 0: 0.044*"object" + 0.033*"snake" + 0.033*"cruciatus" + 0.033*"pain" + 0.022*"repels" + 0.022*"spider" + 0.022*"lumos" + 0.022*"produce" + 0.022*"death" + 0.022*"killing"
Topic 1: 0.062*"protego" + 0.047*"shield" + 0.047*"mend" + 0.047*"patronus" + 0.047*"heals" + 0.047*"episkey" + 0.031*"explosion" + 0.016*"duri" + 0.016*"enemy" + 0.016*"fianto"
Topic 2: 0.045*"turn" + 0.023*"block" + 0.023*"reflects" + 0.023*"spell" + 0.023*"physical" + 0.023*"barrier" + 0.023*"protective" + 0.023*"large" + 0.023*"boggart" + 0.023*"boggartbanishing"
Topic 3: 0.039*"stop" + 0.039*"movement" + 0.039*"human" + 0.039*"presence" + 0.039*"memory" + 0.039*"conjures" + 0.020*"nothing" + 0.020*"supposedly" + 0.020*"revealing" + 0.020*"pixy"


In [231]:
# Store dominant topic for each spell
dominant_topics = []

# Store full topic probabilities
topic_distributions = []

for bow in corpus:

    # Get topic probabilities
    topic_probs = lda_model.get_document_topics(bow)

    # Save full probability distribution
    topic_distributions.append(topic_probs)

    # Get dominant topic
    dominant_topic = max(topic_probs, key=lambda x: x[1])

    dominant_topics.append(dominant_topic[0])

# Add dominant topic column to dataframe
df["Dominant Topic"] = dominant_topics
print(df[["Spell Name", "Dominant Topic"]].head(20))

                          Spell Name  Dominant Topic
0                    Summoning Charm               1
1                 Water-Making Spell               3
2   Launch an object up into the air               3
3                    Unlocking Charm               2
4             Spider repelling spell               0
5                      Slowing Charm               3
6                      Killing Curse               3
7                    Exploding Charm               1
8                    Brackium Emendo               1
9                      Cistem Aperio               3
10                     Locking Spell               0
11                    Blasting Curse               1
12                   Cruciatus Curse               0
13                    Severing Charm               0
14                        Dissendium               1
15                 Engorgement Charm               1
16                           Episkey               1
17                    Patronus Charm          

In [232]:
# LDA-only topic labeling with unique category assignment across all topics.
def topic_profile(topic_id: int) -> dict:
    topic_rows = df[df["Dominant Topic"] == topic_id]
    if topic_rows.empty:
        return {
            "topic_id": topic_id,
            "offensive": 0,
            "defensive": 0,
            "control": 0,
            "reveal": 0,
        }

    type_col_candidates = [column for column in df.columns if str(column).strip().lower() == "type"]
    type_col = type_col_candidates[0] if type_col_candidates else None

    topic_text = (
        topic_rows["Spell Name"].fillna("").str.lower()
        + " "
        + topic_rows["Effect"].fillna("").str.lower()
    )
    if type_col:
        topic_text = topic_text + " " + topic_rows[type_col].fillna("").str.lower()

    offensive_markers = ["imperius", "imperio", "cruciatus", "avada", "killing", "kill", "death", "torture", "pain", "dark"]
    defensive_markers = ["heal", "healing", "episkey", "patronus", "shield", "protect", "protego", "counter", "mend", "disarm"]
    control_markers = ["stun", "slow", "bind", "immobil", "control", "stop", "freeze", "repel"]
    reveal_markers = ["reveal", "reveals", "secret", "light", "lumos", "track", "presence", "detect"]

    offensive_score = sum(topic_text.str.contains(marker, regex=False).sum() for marker in offensive_markers)
    defensive_score = sum(topic_text.str.contains(marker, regex=False).sum() for marker in defensive_markers)
    control_score = sum(topic_text.str.contains(marker, regex=False).sum() for marker in control_markers)
    reveal_score = sum(topic_text.str.contains(marker, regex=False).sum() for marker in reveal_markers)

    return {
        "topic_id": topic_id,
        "offensive": int(offensive_score),
        "defensive": int(defensive_score),
        "control": int(control_score),
        "reveal": int(reveal_score),
    }

profiles = [topic_profile(topic_id) for topic_id in range(lda_model.num_topics)]
profiles_df = pd.DataFrame(profiles)

# Enforce unique top-level categories: exactly one Offensive and one Defensive topic.
offensive_topic = profiles_df.sort_values(["offensive", "defensive"], ascending=[False, True]).iloc[0]["topic_id"]
remaining = profiles_df[profiles_df["topic_id"] != offensive_topic]
defensive_topic = remaining.sort_values(["defensive", "offensive"], ascending=[False, True]).iloc[0]["topic_id"]
remaining = remaining[remaining["topic_id"] != defensive_topic]

# Split remaining topics into two distinct utility-style labels.
control_topic = remaining.sort_values(["control", "reveal"], ascending=[False, False]).iloc[0]["topic_id"]
reveal_topic = remaining[remaining["topic_id"] != control_topic].iloc[0]["topic_id"]

topic_labels = {
    int(offensive_topic): f"LDA Topic {int(offensive_topic)}: Offensive/Dark",
    int(defensive_topic): f"LDA Topic {int(defensive_topic)}: Defensive/Healing",
    int(control_topic): f"LDA Topic {int(control_topic)}: Control/Utility",
    int(reveal_topic): f"LDA Topic {int(reveal_topic)}: Revealing/Other",
}

df["Topic Label"] = df["Dominant Topic"].map(topic_labels)

In [236]:
print(df[["Spell Name", "Dominant Topic", "Topic Label"]].head(15))


                          Spell Name  Dominant Topic  \
0                    Summoning Charm               1   
1                 Water-Making Spell               3   
2   Launch an object up into the air               3   
3                    Unlocking Charm               2   
4             Spider repelling spell               0   
5                      Slowing Charm               3   
6                      Killing Curse               3   
7                    Exploding Charm               1   
8                    Brackium Emendo               1   
9                      Cistem Aperio               3   
10                     Locking Spell               0   
11                    Blasting Curse               1   
12                   Cruciatus Curse               0   
13                    Severing Charm               0   
14                        Dissendium               1   

                       Topic Label  
0   LDA Topic 1: Defensive/Healing  
1      LDA Topic 3: Offensive

In [243]:
# show me all the spells with dominant topic
print(df[df["Dominant Topic"] == 3][["Spell Name", "Dominant Topic", "Topic Label"]])


                          Spell Name  Dominant Topic  \
1                 Water-Making Spell               3   
2   Launch an object up into the air               3   
5                      Slowing Charm               3   
6                      Killing Curse               3   
9                      Cistem Aperio               3   
19                     Expulso Curse               3   
21    Human Presence Revealing Spell               3   
22                    Freezing Charm               3   
23                   Impediment Jinx               3   
26                 Fire-Making Spell               3   
35                      Memory Charm               3   
38             Peskipiksi Pesternomi               3   
46                   Shrinking Charm               3   
55                      Sectumsempra               3   

                    Topic Label  
1   LDA Topic 3: Offensive/Dark  
2   LDA Topic 3: Offensive/Dark  
5   LDA Topic 3: Offensive/Dark  
6   LDA Topic 3: Offens

In [235]:
type_col_candidates = [column for column in df.columns if str(column).strip().lower() == "type"]
view_columns = ["Spell Name", "Effect", "Dominant Topic", "Topic Label"]
if type_col_candidates:
    view_columns.insert(1, type_col_candidates[0])

print(
    df[df["Spell Name"].isin(["Episkey", "Cruciatus Curse", "Killing Curse", "Imperius Curse"])][view_columns]
)

         Spell Name                         Effect  Dominant Topic  \
6     Killing Curse            Instantaneous death               3   
12  Cruciatus Curse              Excruciating pain               0   
16          Episkey           Heals minor injuries               1   
24   Imperius Curse  Total control over the victim               2   

                       Topic Label  
6      LDA Topic 3: Offensive/Dark  
12    LDA Topic 0: Control/Utility  
16  LDA Topic 1: Defensive/Healing  
24    LDA Topic 2: Revealing/Other  
